# TODO

In [3]:
## TODO: add visibility example

## TODO:check if there is an image for sample and count the available pixel

## TODO: count by category to evaluate and correctly interprete. for example at 20 samples

## derive object speed

## TODO: may be rename ann_df?

## TODO: for complex code writhe the conceptual algo in plain text


# Objective

The task considered in this project is **3D object detection on the nuScenes dataset**, with a particular focus on the role of **sensor fusion**. For each scene, the objective is to detect relevant traffic participants and estimate their **class, 3D position, size, and orientation**.

Because nuScenes provides multiple sensing modalities, the problem is not only about achieving accurate object detection, but also about understanding how **LiDAR, cameras, and radar** contribute differently to perception. **LiDAR** provides strong geometric and localization cues, **cameras** provide semantic and appearance information, and **radar** can contribute complementary motion-related and range information.

The exploratory data analysis is therefore designed to study detection difficulty across **object classes, distance, visibility level, and scene conditions**, and to identify the regimes in which fusion is likely to provide meaningful benefit over unimodal baselines.

**Goal:** place the **correct 3D bounding box** around each relevant object in the environment surrounding the ego vehicle.

# Setup and imports

In [36]:
# Setup and imports
import os
import sys
import logging
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple  # required for Python 3.8

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from nuscenes.nuscenes import NuScenes
from IPython.display import display
from tqdm.auto import tqdm

In [37]:
# Notebook display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [38]:
# Environment check
print(f"Python version: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Python version: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
CUDA available: True
CUDA device count: 1
GPU: NVIDIA GeForce RTX 4090


# Data loading

In [39]:
# Dataset configuration
RUNNING_ON: str = "ubelix"   # "local" or "ubelix"
DATASET: str = "full"        # "mini" or "full"

In [40]:
# Resolve dataset path and version
def get_nuscenes_config(running_on: str, dataset: str) -> Tuple[str, str]:
    if running_on == "local":
        if dataset == "mini":
            return "../data/sets/nuscenes", "v1.0-mini"
        if dataset == "full":
            return "../data/sets/nuscenes_full", "v1.0-trainval"

    if running_on == "ubelix":
        if dataset == "mini":
            return "/storage/homefs/ae04q066/datasets/nuscenes_mini", "v1.0-mini"
        if dataset == "full":
            return "/rs_scratch/users/ae04q066/nuscenes_full", "v1.0-trainval"

    raise ValueError(
        "Invalid combination: RUNNING_ON must be 'local' or 'ubelix', "
        "and DATASET must be 'mini' or 'full'"
    )

DATAROOT, VERSION = get_nuscenes_config(RUNNING_ON, DATASET)

if not os.path.isdir(DATAROOT):
    raise FileNotFoundError(f"DATAROOT not found: {os.path.abspath(DATAROOT)}")

print(f"RUNNING_ON: {RUNNING_ON}")
print(f"DATASET:    {DATASET}")
print(f"DATAROOT:   {DATAROOT}")
print(f"VERSION:    {VERSION}")

RUNNING_ON: ubelix
DATASET:    full
DATAROOT:   /rs_scratch/users/ae04q066/nuscenes_full
VERSION:    v1.0-trainval


In [41]:
# Load nuScenes
nusc: NuScenes = NuScenes(
    version=VERSION,
    dataroot=str(DATAROOT),
    verbose=True,
) 

# Quick dataset summary
print(f"Loaded version: {nusc.version}")
print(f"Number of logs: {len(nusc.log)}")
print(f"Number of scenes: {len(nusc.scene)}")
print(f"Available tables: {nusc.table_names}")

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 28.541 seconds.
Reverse indexing ...
Done reverse indexing in 6.1 seconds.
Loaded version: v1.0-trainval
Number of logs: 68
Number of scenes: 850
Available tables: ['category', 'attribute', 'visibility', 'instance', 'sensor', 'calibrated_sensor', 'ego_pose', 'log', 'scene', 'sample', 'sample_data', 'sample_annotation', 'map']


# Dataset overview

This section introduces the structure of the nuScenes dataset and the main tables used in the project. Since nuScenes is organized as a relational database, understanding the links between tables is important before starting preprocessing and feature engineering.

The analysis focuses especially on the tables related to **samples**, **sensor data**, and **object annotations**, since these are the core components of the 3D detection task.

Relevant links:
* https://www.nuscenes.org/nuscenes
* https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/schema_nuscenes.md
* https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb

## Database schema

nuScenes is organized as a set of linked tables describing scenes, sensor measurements, object instances, and annotations.

The most relevant tables for this project are:

1. `scene` - a roughly 20-second driving segment.
2. `sample` - an annotated snapshot of a scene at a particular timestamp.
3. `sample_data` - sensor data recorded for one sensor at one sample.
4. `sample_annotation` - one annotated object in a given sample.
5. `instance` - a physical object tracked over time.
6. `category` - the semantic class of an object.
7. `attribute` - time-varying properties of an object.
8. `visibility` - visibility level of an annotation.
9. `ego_pose` - ego vehicle pose at a timestamp.
10. `sensor` and `calibrated_sensor` - sensor definitions and calibration.
11. `log` and `map` - recording context and map information.

The schema below is adapted from the official nuScenes tutorial and documentation.

**Source:** Official nuScenes tutorial and schema documentation.

![](https://www.nuscenes.org/public/images/nuscenes-schema.svg)

## Load nuScenes tables into pandas

To make exploration easier, the nuScenes tables are loaded into a dictionary of pandas DataFrames. This provides a tabular view of the relational structure and simplifies later preprocessing steps.

In [42]:
# Load nuScenes tables into a dictionary of DataFrames
nusc_tables: Dict[str, pd.DataFrame] = {
    table_name: pd.DataFrame(getattr(nusc, table_name))
    for table_name in nusc.table_names
}

table_summary_df: pd.DataFrame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
        }
        for table_name, df in nusc_tables.items()
    ]
).sort_values("table_name").reset_index(drop=True)

table_summary_df

,table_name,n_rows,n_cols
0,attribute,8,3
1,calibrated_sensor,10200,5
2,category,23,3
3,ego_pose,2631083,4
4,instance,64386,5
5,log,68,6
6,map,4,5
7,sample,34149,7
8,sample_annotation,1166187,13
9,sample_data,2631083,14


In [44]:
# High-level dataset inspection
scene_df: pd.DataFrame = nusc_tables["scene"]
category_df: pd.DataFrame = nusc_tables["category"]
attribute_df: pd.DataFrame = nusc_tables["attribute"]

print(f"Number of scenes: {len(scene_df)}")
display(scene_df[["name", "description"]].head())

print(f"\nNumber of categories: {len(category_df)}")
display(category_df[["name", "description"]].head())

print(f"\nNumber of attributes: {len(attribute_df)}")
display(attribute_df[["name", "description"]].head())

Number of scenes: 850


,name,description
0,scene-0001,"Construction, maneuver between several trucks"
1,scene-0002,"Intersection, peds, waiting vehicle, parked motorcycle at parking lot"
2,scene-0003,"Parking lot, barrier, exit parking lot"
3,scene-0004,"Random scene, arrive at intersection, cross intersection, jaywalkers"
4,scene-0005,"Overtaken by taxi, construction site"



Number of categories: 23


,name,description
0,human.pedestrian.adult,Adult subcategory.
1,human.pedestrian.child,Child subcategory.
2,human.pedestrian.wheelchair,"Wheelchairs. If a person is in the wheelchair, include in the annotation."
3,human.pedestrian.stroller,"Strollers. If a person is in the stroller, include in the annotation."
4,human.pedestrian.personal_mobility,"A small electric or self-propelled vehicle, e.g. skateboard, segway, or scooters, on which the person typically travels in a upright position. Driver and (if applicable) rider should be included in the bounding box along with the vehicle."



Number of attributes: 8


,name,description
0,vehicle.moving,Vehicle is moving.
1,vehicle.stopped,"Vehicle, with a driver/rider in/on it, is currently stationary but has an intent to move."
2,vehicle.parked,Vehicle is stationary (usually for longer duration) with no immediate intent to move.
3,cycle.with_rider,There is a rider on the bicycle or motorcycle.
4,cycle.without_rider,There is NO rider on the bicycle or motorcycle.


## Table relationships relevant for 3D detection

The following relationships are especially important for this project:

1. `scene` -> many `sample`
2. `sample` -> one annotated timestamp
3. `sample_data` -> one sensor measurement for one sample
4. `sample_annotation` -> one object annotation for one sample

In other words, a sample represents a single moment in time, `sample_data` stores the sensor-specific observations at that moment, and `sample_annotation` stores the annotated objects associated with that sample.

In [45]:
# Inspect one sample and its associated sensor data tokens
sample_df: pd.DataFrame = nusc_tables["sample"]

sample_data_tokens: Dict[str, str] = sample_df.loc[0, "data"]
pprint(sample_data_tokens)

sample_data_token: Optional[str] = sample_data_tokens.get("CAM_BACK")
print("CAM_BACK token:", sample_data_token)

{'CAM_BACK': 'aab35aeccbda42de82b2ff5c278a0d48',
 'CAM_BACK_LEFT': '86e6806d626b4711a6d0f5015b090116',
 'CAM_BACK_RIGHT': 'ec7096278e484c9ebe6894a2ad5682e9',
 'CAM_FRONT': '020d7b4f858147558106c504f7f31bef',
 'CAM_FRONT_LEFT': '24332e9c554a406f880430f17771b608',
 'CAM_FRONT_RIGHT': '16d39ff22a8545b0a4ee3236a0fe1c20',
 'LIDAR_TOP': '3388933b59444c5db71fade0bbfef470',
 'RADAR_BACK_LEFT': '05fc4678025246f3adf8e9b8a0a0b13b',
 'RADAR_BACK_RIGHT': '31b8099fb1c44c6381c3c71b335750bb',
 'RADAR_FRONT': 'bddd80ae33ec4e32b27fdb3c1160a30e',
 'RADAR_FRONT_LEFT': '1a08aec0958e42ebb37d26612a2cfc57',
 'RADAR_FRONT_RIGHT': '282fa8d7a3f34b68b56fb1e22e697668'}
CAM_BACK token: aab35aeccbda42de82b2ff5c278a0d48


In [46]:
# Inspect the sample_data table
sample_data_df: pd.DataFrame = nusc_tables["sample_data"]
sample_data_df.head()

,token,sample_token,ego_pose_token,calibrated_sensor_token,timestamp,fileformat,is_key_frame,height,width,filename,prev,next,sensor_modality,channel
0,bddd80ae33ec4e32b27fdb3c1160a30e,e93e98b63d3b40209056d129dc53ceee,bddd80ae33ec4e32b27fdb3c1160a30e,7781065816974801afc4dcdaf6acf92c,1531883530440378,pcd,True,0,0,samples/RADAR_FRONT/n015-2018-07-18-11-07-57+0800__RADAR_FRONT__1531883530440378.pcd,,90df03ad4710427aabb5f88fe049df2e,radar,RADAR_FRONT
1,90df03ad4710427aabb5f88fe049df2e,14d5adfe50bb4445bc3aa5fe607691a8,90df03ad4710427aabb5f88fe049df2e,7781065816974801afc4dcdaf6acf92c,1531883530510442,pcd,False,0,0,sweeps/RADAR_FRONT/n015-2018-07-18-11-07-57+0800__RADAR_FRONT__1531883530510442.pcd,bddd80ae33ec4e32b27fdb3c1160a30e,28e6129458c745dbbcbdbb3329e91575,radar,RADAR_FRONT
2,28e6129458c745dbbcbdbb3329e91575,14d5adfe50bb4445bc3aa5fe607691a8,28e6129458c745dbbcbdbb3329e91575,7781065816974801afc4dcdaf6acf92c,1531883530585295,pcd,False,0,0,sweeps/RADAR_FRONT/n015-2018-07-18-11-07-57+0800__RADAR_FRONT__1531883530585295.pcd,90df03ad4710427aabb5f88fe049df2e,d6dbfa9130d247688674417e01121f63,radar,RADAR_FRONT
3,c8a5809f46ec40a4b054849f369c1fb7,14d5adfe50bb4445bc3aa5fe607691a8,c8a5809f46ec40a4b054849f369c1fb7,7781065816974801afc4dcdaf6acf92c,1531883530810656,pcd,False,0,0,sweeps/RADAR_FRONT/n015-2018-07-18-11-07-57+0800__RADAR_FRONT__1531883530810656.pcd,a6adc676620247378c395ce5cc303124,fc83a9a0666448bd8756db15ff491374,radar,RADAR_FRONT
4,a6adc676620247378c395ce5cc303124,14d5adfe50bb4445bc3aa5fe607691a8,a6adc676620247378c395ce5cc303124,7781065816974801afc4dcdaf6acf92c,1531883530735051,pcd,False,0,0,sweeps/RADAR_FRONT/n015-2018-07-18-11-07-57+0800__RADAR_FRONT__1531883530735051.pcd,d6dbfa9130d247688674417e01121f63,c8a5809f46ec40a4b054849f369c1fb7,radar,RADAR_FRONT


## Annotation table and detection targets

For the 3D detection task, the central table is `sample_annotation`. Each row corresponds to one annotated object in one sample.

The detection target is given by the object category, stored in the `category_name` field. In addition to the class label, this table contains the information needed for 3D detection, including object location, size, orientation, and support from LiDAR and radar points.

Key fields in `sample_annotation` include:

- `sample_token`: links the annotation to a sample
- `instance_token`: links repeated annotations of the same object over time
- `category_name`: semantic class of the object
- `visibility_token`: visibility level
- `translation`: 3D box center
- `size`: box dimensions
- `rotation`: box orientation
- `num_lidar_pts`: number of LiDAR points inside the box
- `num_radar_pts`: number of radar points inside the box
- `prev` / `next`: temporal links to the same object in adjacent frames

In [47]:
# Load and rename the annotation table

sample_annotation_df: pd.DataFrame = nusc_tables["sample_annotation"].rename(
    columns={
        "token": "annotation_token",
        "prev": "prev_ann",
        "next": "next_ann",
    }
)

sample_annotation_df.head()

,annotation_token,sample_token,instance_token,visibility_token,attribute_tokens,translation,size,rotation,prev_ann,next_ann,num_lidar_pts,num_radar_pts,category_name
0,2cd832644d09479389ed0785e5de85c9,c36eb85918a84a788e236f5c9eef2b05,5e2b6fd1fab74d04a79eefebbec357bb,3,[],"[993.884, 612.441, 0.675]","[0.3, 0.291, 0.734]","[-0.04208490861058176, 0.0, 0.0, 0.9991140377690821]",5cd018cb2448415ab8aff4dc7256999a,,2,0,movable_object.trafficcone
1,6babb6b4d3ac4dc183091df3cc9a6f8a,4dfab70f0b3f405bbf17f7c534d2dd91,5e2b6fd1fab74d04a79eefebbec357bb,2,[],"[994.016, 612.335, 0.775]","[0.3, 0.291, 0.734]","[-0.04208490861058176, 0.0, 0.0, 0.9991140377690821]",54cb06c51f774827a32aeec481bdee9b,68df06a01e31417fac56e0348940601c,1,0,movable_object.trafficcone
2,35034272eb1f413187ae7b6affb6ec7a,14d5adfe50bb4445bc3aa5fe607691a8,5e2b6fd1fab74d04a79eefebbec357bb,4,[],"[994.029, 612.524, 0.725]","[0.3, 0.291, 0.734]","[-0.04208490861058176, 0.0, 0.0, 0.9991140377690821]",173a50411564442ab195e132472fde71,27081526fc9345f49bb7350f30324fc3,2,0,movable_object.trafficcone
3,173a50411564442ab195e132472fde71,e93e98b63d3b40209056d129dc53ceee,5e2b6fd1fab74d04a79eefebbec357bb,4,[],"[994.031, 612.51, 0.728]","[0.3, 0.291, 0.734]","[-0.04208490861058176, 0.0, 0.0, 0.9991140377690821]",,35034272eb1f413187ae7b6affb6ec7a,2,0,movable_object.trafficcone
4,68df06a01e31417fac56e0348940601c,140c678882194ae69e47136c81168b6d,5e2b6fd1fab74d04a79eefebbec357bb,3,[],"[994.016, 612.335, 0.725]","[0.3, 0.291, 0.734]","[-0.04208490861058176, 0.0, 0.0, 0.9991140377690821]",6babb6b4d3ac4dc183091df3cc9a6f8a,64d29d7fbee54de2b308c778b4ffab99,2,0,movable_object.trafficcone


## Detection classes

The object classes to detect are given by the `category_name` field in the annotation table.

In [48]:
# Inspect the detection classes

class_names: List[str] = sorted(sample_annotation_df["category_name"].unique().tolist())

print("Number of classes:", len(class_names))
print("\nList of classes:")
class_names

Number of classes: 23

List of classes:


['animal',
 'human.pedestrian.adult',
 'human.pedestrian.child',
 'human.pedestrian.construction_worker',
 'human.pedestrian.personal_mobility',
 'human.pedestrian.police_officer',
 'human.pedestrian.stroller',
 'human.pedestrian.wheelchair',
 'movable_object.barrier',
 'movable_object.debris',
 'movable_object.pushable_pullable',
 'movable_object.trafficcone',
 'static_object.bicycle_rack',
 'vehicle.bicycle',
 'vehicle.bus.bendy',
 'vehicle.bus.rigid',
 'vehicle.car',
 'vehicle.construction',
 'vehicle.emergency.ambulance',
 'vehicle.emergency.police',
 'vehicle.motorcycle',
 'vehicle.trailer',
 'vehicle.truck']

# Data prerocessing

In [ ]:
enriched_sa_df: pd.DataFrame = sample_annotation_df.copy()

## Visibility level
Object visibility tells us how much of the object can be seen by the cameras.

In [ ]:
visibility_df: pd.DataFrame = nusc_tables["visibility"]
visibility_df.head()

In [ ]:
# Let's add the visibility level to the DataFrame
token_to_level: Dict[str, str] = visibility_df.set_index("token")["level"].to_dict()
print("map: token to level:", token_to_level)
enriched_sa_df["visibility_level"] = enriched_sa_df["visibility_token"].apply(lambda x: token_to_level.get(x))
enriched_sa_df[["visibility_token", "visibility_level", "category_name"]].head()

In [ ]:
visibility_table: pd.DataFrame = enriched_sa_df.groupby(["category_name","visibility_level"])["visibility_level"].size().unstack(fill_value=0)
visibility_table

In [ ]:
visibility_prop: pd.DataFrame = visibility_table.div(visibility_table.sum(axis=1), axis=0)

visibility_prop_sorted = visibility_prop.sort_values(
    by="v0-40",
    ascending=False
)

ax = visibility_prop_sorted.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6)
)

plt.title("Visibility level proportions by category")
plt.xlabel("Category")
plt.ylabel("Proportion")
plt.xticks(rotation=90, ha="right")

ax.legend(
    title="Visibility level",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## Box size
box size in meters as width, length, height

In [ ]:
# Let's split the size list into width, length, height.
#from documentaiton: "size":Bounding box size in meters: width, length, height.
size_cols: List = ["size_w", "size_l", "size_h"]

for idx, col in enumerate(size_cols):
    enriched_sa_df[col]=enriched_sa_df["size"].apply(lambda x: x[idx])

enriched_sa_df[["size","size_w", "size_l", "size_h"]].head()

In [ ]:
def plot_size_by_class(ann_df):
    med = (
        ann_df.groupby("category_name")[["size_w", "size_l", "size_h"]]
        .median()
        .sort_values("size_l")
    )
    med.plot(kind="bar", figsize=(12, 8))
    plt.title("Median object dimensions by class")
    plt.ylabel("Meters")
    plt.xlabel("Class")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()

plot_size_by_class(enriched_sa_df)

## Translation
Sample annotation table: 
* "translation":             <float> [3] -- Bounding box location in meters as center_x, center_y, center_z.

In [ ]:
# # Create a new DataFrame
# features_df: pd.DataFrame = enriched_sa_df.copy()

# sample token -> lidar token
sample_to_lidar: pd.DataFrame = nusc_tables["sample"][["token", "data"]].copy()
sample_to_lidar["lidar_token"] = sample_to_lidar["data"].apply(
    lambda x: x.get("LIDAR_TOP") if isinstance(x, dict) else None
)
sample_to_lidar: pd.Series = sample_to_lidar.set_index("token")["lidar_token"]

# lidar token -> ego pose token
lidar_to_ego_pose: pd.Series = nusc_tables["sample_data"].set_index("token")["ego_pose_token"]

# ego pose token -> ego vehicle position
ego_pose_to_position: pd.Series = nusc_tables["ego_pose"].set_index("token")["translation"]

# Add the lidar token column
enriched_sa_df["lidar_token"] = enriched_sa_df["sample_token"].map(sample_to_lidar)

# Add the ego pose token column
enriched_sa_df["ego_pose_token"] = enriched_sa_df["lidar_token"].map(lidar_to_ego_pose)

# Add the ego vehicle position column
enriched_sa_df["ego_position"] = enriched_sa_df["ego_pose_token"].map(ego_pose_to_position)

# Add the ego_x, ego_y, ego_z columns
enriched_sa_df[["ego_x", "ego_y", "ego_z"]] = pd.DataFrame(
    enriched_sa_df["ego_position"].tolist(),
    index=enriched_sa_df.index
)

# Add the object position columns x, y, z
# "translation": Bounding box location in meters as center_x, center_y, center_z.
enriched_sa_df[["x", "y", "z"]] = pd.DataFrame(
    enriched_sa_df["translation"].tolist(),
    index=enriched_sa_df.index
)

enriched_sa_df[["translation", "x", "y", "z", "ego_x", "ego_y", "ego_z"]].head()

In [ ]:
enriched_sa_df.head()

# Feature engineering

In [21]:
features_df: pd.DataFrame = enriched_sa_df.copy()

## 	2D Distance from Object to Ego Vehicle

In [22]:
# Add the lidar token column
features_df["lidar_sd_token"] = features_df["sample_token"].map(sample_to_lidar)

# Add the ego pose token column
features_df["ego_pose_token"] = features_df["lidar_sd_token"].map(lidar_to_ego_pose)

# Add the ego vehicle position column
features_df["ego_translation"] = features_df["ego_pose_token"].map(ego_pose_to_position)

# Add the 2D distance to the ego vehicle column
features_df["distance_ego_2d"] = np.sqrt(
    (features_df["x"] - features_df["ego_x"])**2 +
    (features_df["y"] - features_df["ego_y"])**2
)

# Add the 3D distance to the ego vehicle column
features_df["distance_ego_3d"] = np.sqrt(
    (features_df["x"] - features_df["ego_x"])**2 +
    (features_df["y"] - features_df["ego_y"])**2 +
    (features_df["z"] - features_df["ego_z"])**2
)

In [23]:
distance_bins = [0, 10, 20, 30, 40, 50, 70, 100, float("inf")]
distance_labels = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-70", "70-100", "100+"]

features_df["distance_bin"] = pd.cut(
    features_df["distance_ego_2d"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True,
    right=False
)

In [ ]:
features_df[
    [
        "category_name", "translation", "x", "y", "z",
        "ego_x", "ego_y", "ego_z",
        "distance_ego_2d", "distance_ego_3d", "distance_bin"
    ]
].head()

## Box volume

In [ ]:
features_df["volume"]=features_df["size"].apply(lambda x: x[0]*x[1]*x[2])
features_df[["size","volume"]].head()

## Aspect ratio

In [26]:
n_zero = (features_df["size_w"] == 0).sum()
if n_zero > 0:
    print(f"Warning: {n_zero} rows have size_w = 0. aspect_ratio_l_w set to NaN for these rows.")

features_df["aspect_ratio_l_w"] = np.where(
    features_df["size_w"] == 0,
    np.nan,
    features_df["size_l"] / features_df["size_w"]
)

In [ ]:
features_df[["size_l","size_w","aspect_ratio_l_w"]].head()

## LiDAR

* ```has_lidar```: the object is supported by LiDAR, meaning at least some LiDAR points fall on it.
* ```low_lidar_support```: the object has LiDAR support, but it is weak, meaning only a small number of LiDAR points fall on it.

In [ ]:
# ============================================================
# Plot: LiDAR point count distribution per annotation (51 to 200)
#
# Question answered:
# How common are mid-range LiDAR point counts, beyond the sparse-support region?
# ============================================================

import matplotlib.pyplot as plt

def plot_point_count_distribution(df, col, sensor_name, max_points=200):
    pts_freq = (
        df[col]
        .value_counts()
        .sort_index()
        .loc[:max_points]
    )

    plt.figure(figsize=(12, 5))
    plt.bar(pts_freq.index, pts_freq.values)
    plt.xlabel(f"Number of {sensor_name} points")
    plt.ylabel("Number of annotations")
    plt.title(f"{sensor_name} point count distribution per annotation (0 to {max_points})")
    plt.show()
    
plot_point_count_distribution(features_df, "num_lidar_pts", "LiDAR", max_points=200)

num_lidar_pts is not normally distributed.

In [ ]:
features_df.groupby("category_name")["num_lidar_pts"].median().sort_values(ascending=False)

In [ ]:
# TODO: refactor and create a function 
lidar_support_summary = features_df.groupby("category_name").agg(
    n_objects=("category_name", "size"),
    lidar_mean=("num_lidar_pts", "mean"),
    lidar_median=("num_lidar_pts", "median"),
    pct_no_lidar=("num_lidar_pts", lambda s: (s == 0).mean())
).sort_values("pct_no_lidar", ascending=True)

lidar_support_summary

LiDAR support is much more consistent than radar support across categories, with lower zero-support rates for most object types. However, LiDAR coverage remains strongly class-dependent. Large vehicle categories such as buses, trucks, trailers, and construction vehicles receive much denser LiDAR support, while pedestrians, bicycles, and traffic cones are often represented by only a few points. This suggests that **LiDAR is a strong modality overall**, but its object-level evidence can still be sparse for small or thin classes.

In [ ]:
def print_point_count_intervals(df, col):
    x = df[col]

    print(f"{col} distribution:")
    print(f"0 points   : {(x == 0).mean() * 100:.2f}%")
    print(f"1 point    : {(x == 1).mean() * 100:.2f}%")
    print(f"2 points   : {(x == 2).mean() * 100:.2f}%")
    print(f"3-4 points : {x.between(3, 4).mean() * 100:.2f}%")
    print(f"5-9 points : {x.between(5, 9).mean() * 100:.2f}%")
    print(f"10+ points : {(x >= 10).mean() * 100:.2f}%")
    
print_point_count_intervals(features_df, "num_lidar_pts")


* LiDAR often gives few points.
* About 23% of objects have no LiDAR points,
* and about 30% have only 1 to 4 points.
* So weak LiDAR support is common in this dataset.

In [34]:
# TODO: use sparse LiDAR to usable LiDAR.
features_df["no_lidar_support"] = features_df["num_lidar_pts"] == 0
features_df["low_lidar_support"] = features_df["num_lidar_pts"].between(1, 4)
features_df["usable_lidar_support"] = features_df["num_lidar_pts"] >= 5

In [ ]:
# what percentage of annotations fall into each LiDAR support group?
def plot_support_distribution(
    df: pd.DataFrame,
    no_col: str,
    low_col: str,
    usable_col: str,
    sensor_name: str
) -> None:
    """Plot percentage distribution of no/low/usable support."""
    support = pd.Series({
        f"No {sensor_name}": df[no_col].sum(),
        f"Low {sensor_name}": df[low_col].sum(),
        f"Usable {sensor_name}": df[usable_col].sum()
    })

    support_pct = 100 * support / support.sum()

    plt.figure(figsize=(7, 4))
    plt.bar(support_pct.index, support_pct.values)
    plt.ylabel("Percentage of annotations (%)")
    plt.title(f"Distribution of {sensor_name} support levels")
    plt.ylim(0, 100)
    plt.show()
    
plot_support_distribution(
    df=features_df,
    no_col="no_lidar_support",
    low_col="low_lidar_support",
    usable_col="usable_lidar_support",
    sensor_name="LiDAR"
)

In [ ]:
features_df[["category_name", "num_lidar_pts","no_lidar_support","low_lidar_support","usable_lidar_support"]].head()

## Radar

* ```has_radar```: the object is supported by radar, meaning at least some radar points fall on it.
* ```low_radar_support```: the object has radar support, but it is weak, meaning only a small number of radar points fall on it.

In [ ]:
plot_point_count_distribution(features_df, "num_radar_pts", "Radar", max_points=30)

```num_radar_pts``` is not normally distributed.

In [ ]:
features_df["num_radar_pts"].describe()

In [ ]:
features_df.groupby("category_name")["num_radar_pts"].mean().sort_values(ascending=False)

In [ ]:
radar_support_summary = features_df.groupby("category_name").agg(
    n_objects=("category_name", "size"),
    radar_mean=("num_radar_pts", "mean"),
    radar_median=("num_radar_pts", "median"),
    pct_no_radar=("num_radar_pts", lambda s: (s == 0).mean())
).sort_values("pct_no_radar", ascending=True)

radar_support_summary

Radar support is highly class-dependent. Large vehicle categories such as buses, trailers, trucks, and construction vehicles usually receive multiple radar points, while pedestrians, bicycles, motorcycles, traffic cones, and other small objects are often completely unsupported by radar. For many vulnerable-road-user classes, the median radar point count is zero and the proportion of zero-radar annotations is very high.

* Radar is specialized, not universal
* It works best on large rigid vehicles
* It is often absent for pedestrians and small objects
* Even in common classes like cars, radar is sparse and inconsistent

In [41]:
pct_no_radar_by_cat = (
    features_df.groupby("category_name")["num_radar_pts"]
    .apply(lambda s: (s == 0).mean())
)

def radar_support_group(p):
    if p < 0.4:
        return "radar_well_supported"
    elif p <= 0.7:
        return "radar_partial"
    else:
        return "radar_weak"

features_df["radar_support_group_cat"] = (
    features_df["category_name"].map(pct_no_radar_by_cat).apply(radar_support_group)
)

In [ ]:
features_df["num_radar_pts"].value_counts()

Based on the count of unique value, I can select threshold for ```low_radar_support```

In [ ]:
print_point_count_intervals(features_df, "num_radar_pts")

In [44]:
features_df["usable_radar_support"] = features_df["num_radar_pts"] >= 3
features_df["low_radar_support"] = features_df["num_radar_pts"].between(1, 2)
features_df["no_radar_support"] = features_df["num_radar_pts"] == 0

In [ ]:
plot_support_distribution(
    df=features_df,
    no_col="no_radar_support",
    low_col="low_radar_support",
    usable_col="usable_radar_support",
    sensor_name="Radar"
)

In [ ]:
features_df[["category_name","num_radar_pts", "usable_radar_support","low_radar_support", "no_radar_support"]].tail()

## Camera
I use the nuScenes helper ```get_sample_data()``` from ```nuscenes.py``` to check whether a given object annotation appears in each camera view of the same sample. For each object, I go through the 6 camera images, keep only the views where nuScenes returns a box for that annotation, and record which cameras can see it. For every visible view, I then project the 3D box corners into the image with ```view_points()```, following the same projection idea used in ```geometry_utils.py```, and compute the box area in pixels. This gives me camera features such as how many cameras see the object, which cameras see it, and the maximum and average projected box area across those views.

In [ ]:
features_df.head(2)

### Box area

In [48]:
import pandas as pd
import numpy as np
from nuscenes.utils.geometry_utils import BoxVisibility, view_points

How to get the box area from cameras?
1. get the camera ```sample_data``` token from the ```sample```
2. call ```nusc.get_sample_data(...)```
3. keep only your target annotation box
4. project its 3D corners with ```view_points(...)```
5. take the min/max x and y
6. compute ```width * height```

In [ ]:
help(nusc.get_sample_data)
help(nusc.get)
help(view_points)

In [ ]:
import numpy as np
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import view_points


# Adapted from nuscenes.utils.geometry_utils.box_in_image:
# project box corners into the image and keep only corners with positive depth.
def box_area_px(
    box: Box,
    intrinsic: np.ndarray,
    img_w: int,
    img_h: int,
    verbose: bool = False
) -> float:
    """
    Compute the projected 2D box area in image pixels.

    The box must already be in camera coordinates.

    Parameters
    ----------
    box : Box
        nuScenes Box object in camera coordinates.
    intrinsic : np.ndarray
        Camera intrinsic matrix.
    img_w : int
        Image width in pixels.
    img_h : int
        Image height in pixels.
    verbose : bool, default=False
        Reserved for optional debugging.

    Returns
    -------
    float
        Area in pixels of the smallest 2D rectangle covering the projected
        visible box corners. Returns 0.0 if no corner is in front of the camera.
    """
    corners_3d: np.ndarray = box.corners()
    corners_2d: np.ndarray = view_points(corners_3d, intrinsic, normalize=True)[:2, :]

    in_front: np.ndarray = corners_3d[2, :] > 0.1 # True if a corner is at least 0.1 meter in front of the camera.

    if not np.any(in_front):
        return 0.0

    x: np.ndarray = np.clip(a=corners_2d[0, in_front], a_min=0, a_max=img_w)
    y: np.ndarray = np.clip(a=corners_2d[1, in_front], a_min=0, a_max=img_h)

    width: float = max(0.0, float(x.max() - x.min()))
    height: float = max(0.0, float(y.max() - y.min()))

    return float(width * height)

def get_camera_features(nusc: NuScenes, ann_token: str) -> Dict[str, Any]:
    # Start from one object annotation
    ann: Dict[str, Any] = nusc.get("sample_annotation", ann_token)

    # Find the frame where this object appears
    sample: Dict[str, Any] = nusc.get("sample", ann["sample_token"])

    visible_cam_names: List[str] = []
    visible_cam_areas: List[float] = []

    # Check each camera in that frame
    for cam_name, sd_token in sample["data"].items():
        if not cam_name.startswith("CAM"):
            continue

        # Get image size
        sd: Dict[str, Any] = nusc.get("sample_data", sd_token)
        img_w: int = sd["width"]
        img_h: int = sd["height"]

        # Ask nuScenes for this annotation in this camera
        _, boxes, camera_intrinsic = nusc.get_sample_data(
            sd_token,
            box_vis_level=BoxVisibility.ANY,
            selected_anntokens=[ann_token]
        )

        # If no box comes back, the object is not visible there
        if not boxes:
            continue

        area: float = box_area_px(boxes[0], camera_intrinsic, img_w, img_h)

        visible_cam_names.append(cam_name)
        visible_cam_areas.append(area)

    if visible_cam_areas:
        best_idx: int = int(np.argmax(visible_cam_areas))
        max_box_area_px: float = float(np.max(visible_cam_areas))
        mean_box_area_px: float = float(np.mean(visible_cam_areas))
        best_camera_name: Optional[str] = visible_cam_names[best_idx]
    else:
        max_box_area_px = 0.0
        mean_box_area_px = 0.0
        best_camera_name = None

    return {
        "num_visible_cams": len(visible_cam_names),
        "visible_cam_names": visible_cam_names,
        "has_any_camera_view": len(visible_cam_names) > 0,
        "max_box_area_px": max_box_area_px,
        "mean_box_area_px": mean_box_area_px,
        "best_camera_name": best_camera_name,
    }


# ------------------------------------------------------------
# test on one annotation first
# ------------------------------------------------------------
test_ann_token = features_df.iloc[0]["annotation_token"]

test_result = get_camera_features(nusc, test_ann_token)

print("Test annotation token:", test_ann_token)
pprint(test_result)

In [ ]:
# TODO: Current computation time: 19m26s. Can I optimize the computation time? 

def add_camera_feature_columns(df: pd.DataFrame, nusc) -> pd.DataFrame:
    camera_cols: List[str] = [
        "num_visible_cams",
        "visible_cam_names",
        "has_any_camera_view",
        "max_box_area_px",
        "mean_box_area_px",
        "best_camera_name",
    ]

    df = df.drop(columns=camera_cols, errors="ignore").copy()

    unique_tokens = df["annotation_token"].unique()

    feature_map: Dict[str, Dict[str, Any]] = {
        ann_token: get_camera_features(nusc, ann_token)
        for ann_token in tqdm(unique_tokens, desc="Computing camera features")
    }

    camera_features_df = df["annotation_token"].map(feature_map).apply(pd.Series)
    camera_features_df.index = df.index

    return pd.concat([df, camera_features_df], axis=1)

# For each annotation (1.16M times):
#     → find its sample
#     → loop over 6 cameras
#     → project 3D box into image
#     → compute box area

features_df =  add_camera_feature_columns(features_df, nusc)    

In [ ]:
features_df[
[
    "annotation_token",
    "num_visible_cams",
    "visible_cam_names",
    "has_any_camera_view",
    "max_box_area_px",
    "mean_box_area_px",
    "best_camera_name",
]
].head()

### box features

In [ ]:
features_df.groupby("category_name")["max_box_area_px"].median().sort_values(ascending=False)

In [ ]:
features_df["max_box_area_px"].quantile([0.25, 0.5, 0.75, 0.9])

In [56]:
q25 = features_df["max_box_area_px"].quantile(0.25)
q50 = features_df["max_box_area_px"].quantile(0.50)

features_df["weak_camera_view"] = features_df["max_box_area_px"] <= q25
features_df["limited_camera_view"] = features_df["max_box_area_px"].between(q25, q50, inclusive="right")
features_df["good_camera_view"] = features_df["max_box_area_px"] > q50

In [ ]:
plot_support_distribution(
    df=features_df,
    no_col="weak_camera_view",
    low_col="limited_camera_view",
    usable_col="limited_camera_view",
    sensor_name="camera_view"
)

In [ ]:
features_df.columns

In [ ]:
# ------------------------------------------------------------
# Step 12: inspect the new threshold features
# ------------------------------------------------------------
print(features_df["weak_camera_view"].value_counts())
print(features_df["limited_camera_view"].value_counts())
print(features_df["good_camera_view"].value_counts())

# Exploratory Data Analysis

In [59]:
# # ============================================================
# # Column groups for EDA
# # Keep columns grouped by the question they help answer.
# # ============================================================

# # Object identifiers for traceability and grouping.
# id_cols = [
#     "annotation_token",
#     "sample_token",
#     "instance_token",
# ]

# # Class label and visibility information.
# context_cols = [
#     "category_name",
#     "visibility_level",
# ]

# # Geometry and intrinsic difficulty features.
# geometry_cols = [
#     "size_w",
#     "size_l",
#     "size_h",
#     "volume",
#     "aspect_ratio_l_w",
#     "x",
#     "y",
#     "z",
#     "distance_ego_2d",
#     "distance_ego_3d",
# ]

# # LiDAR support features.
# lidar_cols = [
#     "num_lidar_pts",
#     "no_lidar_support",
#     "low_lidar_support",
#     "usable_lidar_support",
# ]

# # Radar support features.
# radar_cols = [
#     "num_radar_pts",
#     "has_radar",
#     "low_radar_support",
#     "no_radar_support",
#     "radar_support_group_cat",
# ]

# # Camera support features.
# camera_cols = [
#     "num_visible_cams",
#     "has_any_camera_view",
#     "max_box_area_px",
#     "mean_box_area_px",
#     "best_camera_name",
#     "weak_camera_view",
#     "limited_camera_view",
#     "good_camera_view",
#     "multi_camera_object",
#     "single_camera_object",
#     "multi_camera_good_view",
#     "single_camera_weak_view",
# ]

# # Fusion and complementarity flags.
# fusion_cols = [
#     "low_lidar_good_camera",
#     "no_lidar_good_camera",
#     "low_radar_good_camera",
#     "no_radar_good_camera",
#     "weak_camera_usable_lidar",
#     "weak_camera_has_radar",
#     "weak_camera_any_range_support",
#     "weak_all_modalities",
#     "limited_camera_low_lidar_no_radar",
#     "good_camera_usable_lidar",
#     "good_camera_has_radar",
#     "strong_all_modalities",
# ]

# # Final column list for the EDA dataframe.
# eda_cols = (
#     id_cols
#     + context_cols
#     + geometry_cols
#     + lidar_cols
#     + radar_cols
#     + camera_cols
#     + fusion_cols
# )

# eda_df = features_df[eda_cols].copy()

In [60]:
raw_eda_df = features_df.copy()

In [ ]:
plt.figure(figsize=(8, 5))
raw_eda_df["distance_ego_2d"].hist(bins=50)
plt.title("Object Distance from Ego Vehicle")
plt.xlabel("Distance (m)")
plt.ylabel("Count")
plt.show()

## Sanity checks
The dataset is well curated

In [ ]:
# Which objects dominate the dataset and which are underrepresented?
class_counts:pd.Series = raw_eda_df["category_name"].value_counts().sort_values(ascending=False)
class_counts

## Clean the DataFrame for statistical analysis

In [ ]:
keep_category = class_counts[(class_counts>=30)].index
removed_category = class_counts[(class_counts<30)].index

print("Kept category:",keep_category)
print("\nRemoved category due to low count number:",removed_category)

eda_df = raw_eda_df[raw_eda_df["category_name"].isin(keep_category)].copy()

In [64]:
import matplotlib.pyplot as plt

class_counts = eda_df["category_name"].value_counts()

top_n = 6
top_counts = class_counts.iloc[:top_n]
other_count = class_counts.iloc[top_n:].sum()

pie_counts = top_counts.copy()
pie_counts["Other"] = other_count

excluded_categories = class_counts.iloc[top_n:]

In [ ]:
# ============================================================
# Annotation count by category
#
# Question answered:
# Which objects dominate the dataset?
# ============================================================

# number of objects by category_name
class_counts:pd.Series = raw_eda_df["category_name"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(15, 5))
class_counts.plot(kind="bar")
plt.title("Count per category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

class_counts = eda_df["category_name"].value_counts()

top_n = 7
top_counts = class_counts.iloc[:top_n]
other_count = class_counts.iloc[top_n:].sum()

pie_counts = top_counts.copy()
pie_counts["Other"] = other_count

excluded_categories = class_counts.iloc[top_n:]

plt.figure(figsize=(8, 8))
plt.pie(
    pie_counts.values,
    labels=pie_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title(f"Top {top_n} categories vs Other")
plt.tight_layout()
plt.show()

print("\nCategories included in 'Other':")
print(excluded_categories)

**Note:**
* Category frequencies are not uniform, and some classes may be underrepresented.
* This class imbalance can affect the stability of class-wise statistical results.
* Therefore, a minimum count threshold of 30 annotations per class will be used when deciding whether a category is suitable for reliable statistical analysis.
 Categories below this threshold will be excluded from detailed class-level interpretation or clearly marked as low-support.

## Intrinsic object difficulty
To identify intrinsically difficult objects, I will check in this order:

* Rare classes
* Distance by class
* Volume and dimensions by class
* Visibility by class
* Projected box area by class
* Compound hard cases by class

### Object volume by category

In [ ]:
# ============================================================
# Object volume by category
#
# Question answered:
# Which categories are physically small or large?
# ============================================================

def plot_box_by_category(
    df: pd.DataFrame,
    y_col: str,
    title: str,
    ylabel: str,
    x_col: str = "category_name",
    figsize: tuple = (10,10),
    order_by_median: bool = True,
    log_scale: bool = False,
    show_fliers: bool = True,
    rotate_xticks: int = 90
) -> None:
    """Plot a boxplot of one numeric feature by category."""
    plot_df = df.dropna(subset=[x_col, y_col]).copy()

    order = None
    if order_by_median:
        order = (
            plot_df.groupby(x_col)[y_col]
            .median()
            .sort_values()
            .index
        )

    plt.figure(figsize=figsize)
    sns.boxplot(
        data=plot_df,
        x=x_col,
        y=y_col,
        order=order,
        # showfliers=show_fliers # show outliers
    )

    plt.title(title)
    plt.xlabel("Category")
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotate_xticks, ha="right")

    if log_scale:
        plt.yscale("log")

    plt.tight_layout()
    plt.show()   

plot_box_by_category(
    df=eda_df,
    y_col="volume",
    title="Object volume by category",
    ylabel="Volume",
    log_scale=True
)

**Note:**
* Object volume differs strongly across categories and spans multiple orders of magnitude.
* Small classes such as traffic cones and debris lie at the low end, while buses, trucks, and trailers lie at the high end.
* Some categories have tight size distributions, while others, especially large vehicles, show wider spread.
* This suggests that object volume is a useful feature for distinguishing broad category groups, though some mid-size classes still overlap.

In [ ]:
def get_low_classes_by_median(
    df: pd.DataFrame,
    value_col: str,
    category_col: str = "category_name",
    threshold: float = 1.0
) -> pd.DataFrame:
    """Return classes whose median value is below the given threshold."""
    summary_df = (
        df.dropna(subset=[category_col, value_col])
        .groupby(category_col)[value_col]
        .median()
        .sort_values()
        .reset_index()
    )
    summary_df.columns = [category_col, f"median_{value_col}"]

    return summary_df.loc[summary_df[f"median_{value_col}"] < threshold].copy()

low_volume_df = get_low_classes_by_median(
    eda_df,
    value_col="volume",
    threshold=1.0
)
print(low_volume_df)

### Object aspectio ratio by category

In [ ]:
# ============================================================
# Object volume by aspect ratio
#
# Question answered:
# Which categories have small or large aspect ratio?
# ============================================================

plot_box_by_category(
    df=eda_df,
    y_col="aspect_ratio_l_w",
    title="Object aspect_ratio_l_w by category",
    ylabel="aspect_ratio",
    log_scale=True
)

In [ ]:
low_aspect_ratio_df = get_low_classes_by_median(
    eda_df,
    value_col="aspect_ratio_l_w",
    threshold=1.0
)
print(low_volume_df)

### Object visibility by category

In [ ]:
# ============================================================
# Object visibility by category
#
# Question answered:
# Which categories are poorly visible?
# ============================================================

vis_by_class = (
    pd.crosstab(
        index = eda_df["category_name"],
        columns = eda_df["visibility_level"],
        normalize="index",
    )
    .sort_values("v0-40")
)

vis_by_class


In [ ]:
vis_by_class.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Visibility Distribution by Class")
plt.xlabel("Class")
plt.ylabel("Proportion")
plt.legend(title="Visibility", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
low_visibility_df = vis_by_class[vis_by_class["v0-40"]>0.25].copy()
low_visibility_df

### Hard categories summary

In [74]:
low_visibility_set = set(low_visibility_df.index)
low_aspect_ratio_set = set(low_aspect_ratio_df["category_name"])
low_volume_set = set(low_volume_df["category_name"])

In [ ]:
hard_in_all_three = (
    low_visibility_set
    & low_aspect_ratio_set
    & low_volume_set
)

print("\n Hard in all three:")
pprint(hard_in_all_three)

hard_in_at_least_two = (
    (low_visibility_set & low_aspect_ratio_set)
    | (low_visibility_set & low_volume_set)
    | (low_aspect_ratio_set & low_volume_set)
)

print("\n Hard in at least two:")
pprint(hard_in_at_least_two)

In [ ]:
print("Low visibility + low aspect ratio:")
print(low_visibility_set & low_aspect_ratio_set)

print("\nLow visibility + low volume:")
print(low_visibility_set & low_volume_set)

print("\nLow aspect ratio + low volume:")
print(low_aspect_ratio_set & low_volume_set)

In [ ]:
hard_rows = eda_df[eda_df["category_name"].isin(hard_in_at_least_two)].shape[0]
all_rows = eda_df.shape[0]

print("All intrinsic hard categories:\n",hard_in_at_least_two)
print("\nPercentage of rows from hard categories:", hard_rows / all_rows * 100)

### Observations

This section should answer:

which categories are inherently hard because of geometry or visibility
whether those hard categories are marginal or substantial in the dataset
Example key insights
Example 1

Some categories are intrinsically difficult because they combine small physical size, unfavorable shape, and lower visibility, making them challenging independently of sensor support.

Example 2

Intrinsic difficulty is not evenly distributed across classes: a small subset of categories accounts for a large share of the most difficult objects.

Example 3

Categories such as traffic cones, barriers, or pedestrians may appear difficult not only because of scene conditions, but because their geometry and visibility characteristics are already unfavorable.

Example 4

Since intrinsically hard categories cover a substantial fraction of dataset rows, later sensor-regime analysis should not treat them as rare corner cases.

Strong concluding sentence template

Overall, intrinsic difficulty is concentrated in a small number of categories whose object size, shape, and visibility make them systematically harder to detect, suggesting that later fusion analysis should pay particular attention to these classes.

 ## Sensor reliability by regime
This section characterizes the reliability of each sensing modality across distance and visibility regimes, in order to identify where each sensor is informative, weak, or missing before analyzing fusion complementarity.

### Camera reliability by distance

In [78]:
def summarize_flag_by_group(df, group_col, flag_col, sort_index=True):
    table = (
        pd.crosstab(
            index=df[group_col],
            columns=df[flag_col],
            normalize="index"
        )
    )

    if sort_index:
        table = table.sort_index()

    return table

cam_weak_by_distance = summarize_flag_by_group(eda_df, "distance_bin", "weak_camera_view")
cam_limited_by_distance = summarize_flag_by_group(eda_df, "distance_bin", "limited_camera_view")
cam_good_by_distance = summarize_flag_by_group(eda_df, "distance_bin", "good_camera_view")

In [ ]:
def plot_bar_from_table(
    table,
    column=None,
    title="",
    ylabel="",
    xlabel="",
    figsize=(8, 4),
    rot=0,
    ha="center"
):
    """
    Plot a bar chart from a pandas Series or DataFrame.

    """
    if column is not None:
        ax = table[column].plot.bar(figsize=figsize, title=title, rot=rot)
    else:
        ax = table.plot.bar(figsize=figsize, title=title, rot=rot)

    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=rot, ha=ha)
    plt.show()

    return None

camera_flags = {
    "weak_camera_view": "Weak camera view",
    "limited_camera_view": "Limited camera view",
    "good_camera_view": "Good camera view"
}

for flag, label in camera_flags.items():
    table = summarize_flag_by_group(eda_df, "distance_bin", flag)
    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin"
    )

### Camera reliability by visibility

In [80]:
cam_weak_by_visibility = summarize_flag_by_group(eda_df, "visibility_level", "weak_camera_view")
cam_limited_by_visibility = summarize_flag_by_group(eda_df, "visibility_level", "limited_camera_view")
cam_good_by_visibility = summarize_flag_by_group(eda_df, "visibility_level", "good_camera_view")

In [ ]:
for flag, label in camera_flags.items():
    table = summarize_flag_by_group(eda_df, "visibility_level", flag)
    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level"
    )


In [ ]:
plot_bar_from_table(
    table=cam_limited_by_visibility,
    column=True,
    title="Limited camera view by visibility",
    ylabel="Proportion",
    xlabel="Visibility level"
)

### Camera reliability by category

In [84]:
cam_good_by_category = summarize_flag_by_group(eda_df, "category_name", "good_camera_view", sort_index=False)
cam_good_by_category_sorted = cam_good_by_category.sort_values(by=True, ascending=False)

In [ ]:
for flag, label in camera_flags.items():
    table = summarize_flag_by_group(eda_df, "category_name", flag, sort_index=False)
    table = table.sort_values(by=True, ascending=False)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by category",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Category",
        rot=90
    )

### Camera summary table

In [86]:
camera_flags = {
    "weak_camera_view": "Weak camera view",
    "limited_camera_view": "Limited camera view",
    "good_camera_view": "Good camera view"
}


In [ ]:
camera_support_summary_by_distance = (
    eda_df.groupby("distance_bin")[
        ["weak_camera_view", "limited_camera_view", "good_camera_view"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(camera_support_summary_by_distance)

In [ ]:
camera_support_summary_by_distance = (
    eda_df.groupby("visibility_level")[
        ["weak_camera_view", "limited_camera_view", "good_camera_view"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(camera_support_summary_by_distance)

### LiDAR reliability by distance

In [89]:
lidar_flags = {
    "no_lidar_support": "No LiDAR support",
    "low_lidar_support": "Low LiDAR support",
    "usable_lidar_support": "Usable LiDAR support"
}

In [ ]:
for flag, label in lidar_flags.items():
    table = summarize_flag_by_group(eda_df, "distance_bin", flag)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin"
    )

### LiDAR reliability by visibility

In [ ]:
for flag, label in lidar_flags.items():
    table = summarize_flag_by_group(eda_df, "visibility_level", flag)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level"
    )

LiDAR reliability by category

In [ ]:
for flag, label in lidar_flags.items():
    table = summarize_flag_by_group(
        eda_df, "category_name", flag, sort_index=False
    )

    table = table.sort_values(by=True, ascending=False)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by category",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Category",
        rot=90
    )

### LiDAR summary table

In [ ]:
lidar_support_summary_by_distance = (
    eda_df.groupby("distance_bin")[
        ["no_lidar_support", "low_lidar_support", "usable_lidar_support"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(lidar_support_summary_by_distance)

In [ ]:
lidar_support_summary_by_visibility = (
    eda_df.groupby("visibility_level")[
        ["no_lidar_support", "low_lidar_support", "usable_lidar_support"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(lidar_support_summary_by_visibility)

In [ ]:
# ============================================================
# LiDAR support by factor
#
# Question answered:
# When does LiDAR become sparse?
# ============================================================
lidar_low_by_distance = (
    pd.crosstab(
        index=eda_df["distance_bin"],
        columns=eda_df["low_lidar_support"],
        normalize="index",
    )
    .sort_index()
)

lidar_low_by_distance

In [ ]:
plot_bar_from_table(
    table=lidar_low_by_distance,
    column=True,
    title="Low lidar support",
    ylabel="Proportion of lidar_low_by_distance",
    xlabel="Distance"
)

In [ ]:
usable_lidar_support = (
    pd.crosstab(
        index=eda_df["category_name"],
        columns=eda_df["low_lidar_support"],
        normalize="index",
    )
    .sort_index()
)

usable_lidar_support_sorted = usable_lidar_support.sort_values(True).copy()
usable_lidar_support_sorted 

In [ ]:
plot_bar_from_table(
    table=usable_lidar_support_sorted,
    column=True,
    title="Low lidar support by category",
    ylabel="Proportion of lidar_low_by_categroy",
    xlabel="Category name",
    rot=90
)

**Note**:

LiDAR struggles with small, thin, and irregular objects, but performs reliably on large, solid objects.

### Radar reliability by distance

In [99]:
radar_flags = {
    "no_radar_support": "No radar support",
    "low_radar_support": "Low radar support",
    "usable_radar_support": "Usable radar support",
    "usable_radar_support": "Radar present"
}

In [100]:
radar_present_by_distance = summarize_flag_by_group(
    eda_df, "distance_bin", "usable_radar_support"
)

radar_none_by_distance = summarize_flag_by_group(
    eda_df, "distance_bin", "no_radar_support"
)

radar_low_by_distance = summarize_flag_by_group(
    eda_df, "distance_bin", "low_radar_support"
)

radar_usable_by_distance = summarize_flag_by_group(
    eda_df, "distance_bin", "usable_radar_support"
)

In [ ]:
for flag, label in radar_flags.items():
    table = summarize_flag_by_group(eda_df, "distance_bin", flag)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin"
    )

### Radar reliability by visibility

In [ ]:
for flag, label in radar_flags.items():
    table = summarize_flag_by_group(eda_df, "visibility_level", flag)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level"
    )

### Radar reliability by category

In [ ]:
for flag, label in radar_flags.items():
    table = summarize_flag_by_group(
        eda_df, "category_name", flag, sort_index=False
    ).sort_values(by=True, ascending=False)

    plot_bar_from_table(
        table=table,
        column=True,
        title=f"{label} by category",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Category",
        rot=90
    )

### Radar summary table

In [ ]:
radar_support_summary_by_distance = (
    eda_df.groupby("distance_bin")[
        ["no_radar_support", "low_radar_support", "usable_radar_support"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(radar_support_summary_by_distance)

In [ ]:
radar_support_summary_by_visibility = (
    eda_df.groupby("visibility_level")[
        ["no_radar_support", "low_radar_support", "usable_radar_support", "usable_radar_support"]
    ]
    .mean()
    .mul(100)
    .round(1)
)

print(radar_support_summary_by_visibility)

In [ ]:
# ============================================================
# Radar support by factor
#
# Question answered:
# When is radar meaningfully present?
# ============================================================
radar_present_by_distance = (
    pd.crosstab(
        index=eda_df["distance_bin"],
        columns=eda_df["usable_radar_support"],
        normalize="index",
    )
    .sort_index()
)

radar_present_by_distance

In [ ]:
plot_bar_from_table(
    table=radar_present_by_distance,
    column=True,
    title="radar_present_by_distance",
    ylabel="Proportion of radar_present_by_distance",
    xlabel="Distance",
    rot=90
)

In [108]:
radar_present_by_class = (
    pd.crosstab(
        index=eda_df["category_name"],
        columns=eda_df["usable_radar_support"],
        normalize="index",
    )
    .sort_index()
)

radar_present_by_class_sorted = radar_present_by_class.sort_values(True).copy()

In [ ]:
plot_bar_from_table(
    table=radar_present_by_class_sorted,
    column=True,
    title="radar_present_by_class",
    ylabel="Proportion of radar_present_by_class",
    xlabel="Category",
    rot=90
)

In [ ]:
# ============================================================
# Combine camera, LiDAR, and radar category rates in one plot
# ============================================================

modality_by_category = pd.concat(
    [
        cam_good_by_category_sorted[True].rename("good_camera_view"),
        usable_lidar_support_sorted[True].rename("low_lidar_support"),
        radar_present_by_class_sorted[True].rename("has_radar"),
    ],
    axis=1
)

modality_by_category.plot.bar(figsize=(12, 5), rot=90)
plt.title("Single-modality signals by category")
plt.ylabel("Proportion")
plt.xlabel("Category")
plt.show()

### Observations
What this section should conclude

This section should answer:

when camera is reliable
when LiDAR is reliable
when radar is available/useful
how these change with distance and visibility
Example key insights

Camera reliability

* Camera support is strongest in close to mid-range regimes and weakens as object distance increases.
* Camera quality is also sensitive to visibility, with weak-camera cases becoming more common under poorer visibility conditions.
* Categories with small image footprint are more likely to suffer from weak camera support.

LiDAR reliability

* LiDAR support is not uniformly strong; sparse or missing LiDAR occurs in non-trivial parts of the dataset.
* Low LiDAR support may behave more like missing LiDAR than like robust range support.
* LiDAR reliability changes with regime and should not be treated as constant.

Radar reliability

* Radar is often absent or sparse, so it should be viewed as a complementary modality rather than a universally available one.
* When present, radar may remain useful in regimes where camera support weakens, especially at longer range.


Strong concluding sentence template

Overall, sensor reliability is strongly regime-dependent: camera is typically strongest in more favorable visual conditions, while LiDAR and radar provide uneven but potentially valuable support depending on distance, visibility, and object class.

## Cross-modal complementarity

In [111]:
# ============================================================
# Section 3: Cross-modal complementarity
#
# Goal:
# Identify regimes where one modality remains useful while
# another is weak or absent. This is the main evidence for
# whether fusion is actually needed and where it adds value.
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Helper function

In [112]:
def summarize_flags_by_group(df: pd.DataFrame, group_col: str, flag_cols: List[str]) -> pd.DataFrame:
    """Return mean rate of boolean flags by group."""
    return (
        df.groupby(group_col)[flag_cols]
        .mean()
        .sort_index()
    )

def plot_grouped_flag_rates(
    table: pd.DataFrame,
    title: str,
    xlabel: str,
    ylabel: str = "Proportion",
    figsize: tuple = (10, 5),
    rot: int = 0
) -> None:
    """Plot grouped bar chart from grouped flag-rate table."""
    ax = table.plot(kind="bar", figsize=figsize)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=rot)
    plt.legend(
        title="",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )
    plt.tight_layout()
    plt.show()


def plot_flag_by_category(
    df: pd.DataFrame,
    flag_col: str,
    title: str,
    ylabel: str = "Proportion",
    xlabel: str = "Category",
    rot: int = 90,
    ascending: bool = False
) -> pd.DataFrame:
    """Plot normalized per-category rate for one boolean flag."""
    table = (
        pd.crosstab(
            index=df["category_name"],
            columns=df[flag_col],
            normalize="index",
        )
        .sort_values(by=True, ascending=ascending)
    )

    plot_bar_from_table(
        table=table,
        column=True,
        title=title,
        ylabel=ylabel,
        xlabel=xlabel,
        rot=rot
    )
    return table

### Define complementarity flags

In [113]:
# Camera compensates when range sensing is weak
eda_df["good_camera_no_lidar"] = (
    eda_df["good_camera_view"]
    & eda_df["no_lidar_support"]
)

eda_df["good_camera_no_radar"] = (
    eda_df["good_camera_view"]
    & eda_df["no_radar_support"]
)

eda_df["good_camera_weak_range"] = (
    eda_df["good_camera_view"]
    & (
        ~eda_df["usable_lidar_support"]
        | ~eda_df["usable_radar_support"]
    )
)

# Range sensing compensates when camera is weak
eda_df["weak_camera_usable_lidar"] = (
    eda_df["weak_camera_view"]
    & eda_df["usable_lidar_support"]
)

eda_df["weak_camera_has_radar"] = (
    eda_df["weak_camera_view"]
    & eda_df["usable_radar_support"]
)

eda_df["weak_camera_any_range_support"] = (
    eda_df["weak_camera_view"]
    & (
        eda_df["usable_lidar_support"]
        | eda_df["usable_radar_support"]
    )
)

### Global prevalence of complementarity regimes

In [ ]:
# TODO: all the percentage should add up to 100% for better analysis

complementarity_flags = [
    "good_camera_no_lidar",
    "good_camera_no_radar",
    "good_camera_weak_range",
    "weak_camera_usable_lidar",
    "weak_camera_has_radar",
    "weak_camera_any_range_support",
]

complementarity_label_map = {
    "good_camera_no_lidar": "Good camera + no LiDAR",
    "good_camera_no_radar": "Good camera + no radar",
    "good_camera_weak_range": "Good camera + weak range",
    "weak_camera_usable_lidar": "Weak camera + usable LiDAR",
    "weak_camera_has_radar": "Weak camera + radar",
    "weak_camera_any_range_support": "Weak camera + any range support",
}

complementarity_summary = (
    eda_df[complementarity_flags]
    .mean()
    .mul(100)
    .rename(index=complementarity_label_map)
    .sort_values(ascending=False)
    .round(2)
)

print("Complementarity regime prevalence (% of rows):")
print(complementarity_summary)

plt.figure(figsize=(8, 5))
sns.barplot(
    x=complementarity_summary.values,
    y=complementarity_summary.index
)
plt.title("Prevalence of complementarity regimes")
plt.xlabel("Percentage of rows")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Camera compensates weak range support

In [ ]:
camera_help_flags = [
    "good_camera_no_lidar",
    "good_camera_no_radar",
    "good_camera_weak_range",
]

camera_help_labels = {
    "good_camera_no_lidar": "Good camera + no LiDAR",
    "good_camera_no_radar": "Good camera + no radar",
    "good_camera_weak_range": "Good camera + weak range",
}

# By distance
camera_help_by_distance = summarize_flags_by_group(
    eda_df,
    "distance_bin",
    camera_help_flags
).rename(columns=camera_help_labels)

plot_grouped_flag_rates(
    table=camera_help_by_distance,
    title="Camera-helpful regimes by distance",
    xlabel="Distance bin",
    ylabel="Proportion",
    rot=90
)

# By visibility
camera_help_by_visibility = summarize_flags_by_group(
    eda_df,
    "visibility_level",
    camera_help_flags
).rename(columns=camera_help_labels)

plot_grouped_flag_rates(
    table=camera_help_by_visibility,
    title="Camera-helpful regimes by visibility",
    xlabel="Visibility level",
    ylabel="Proportion",
    rot=90
)

### Range sensing compensates weak camera

In [ ]:
range_help_flags = [
    "weak_camera_usable_lidar",
    "weak_camera_has_radar",
    "weak_camera_any_range_support",
]

range_help_labels = {
    "weak_camera_usable_lidar": "Weak camera + usable LiDAR",
    "weak_camera_has_radar": "Weak camera + radar",
    "weak_camera_any_range_support": "Weak camera + any range support",
}

# By distance
range_help_by_distance = summarize_flags_by_group(
    eda_df,
    "distance_bin",
    range_help_flags
).rename(columns=range_help_labels)

plot_grouped_flag_rates(
    table=range_help_by_distance,
    title="Range-helpful regimes by distance",
    xlabel="Distance bin",
    ylabel="Proportion",
    rot=90
)

# By visibility
range_help_by_visibility = summarize_flags_by_group(
    eda_df,
    "visibility_level",
    range_help_flags
).rename(columns=range_help_labels)

plot_grouped_flag_rates(
    table=range_help_by_visibility,
    title="Range-helpful regimes by visibility",
    xlabel="Visibility level",
    ylabel="Proportion",
    rot=90
)

### category-level views

In [ ]:
# ------------------------------------------------------------
# 5. Optional category-level views
# Keep only 1-2 compact views to avoid clutter
# ------------------------------------------------------------

# A. Categories where camera is often good while range is weak
cam_help_by_category = plot_flag_by_category(
    df=eda_df,
    flag_col="good_camera_weak_range",
    title="Good camera but weak range support by category",
    ylabel="Proportion",
    ascending=False
)

# B. Categories where camera is weak but at least one range sensor helps
range_help_by_category = plot_flag_by_category(
    df=eda_df,
    flag_col="weak_camera_any_range_support",
    title="Weak camera but any range support by category",
    ylabel="Proportion",
    ascending=False
)




### compact summary tables for reporting

In [ ]:
# ------------------------------------------------------------
# 6. Optional compact summary tables for reporting
# ------------------------------------------------------------
print("\nCamera-helpful regimes by distance:")
display(camera_help_by_distance.round(3))

print("\nRange-helpful regimes by distance:")
display(range_help_by_distance.round(3))

print("\nCamera-helpful regimes by visibility:")
display(camera_help_by_visibility.round(3))

print("\nRange-helpful regimes by visibility:")
display(range_help_by_visibility.round(3))

### Observations

What this section should conclude

This section should answer:

when camera helps weak range sensing
when range sensors help weak camera
whether complementarity is uniform or regime-specific
Example key insights
Example 1

Complementarity is not uniform across the dataset; it appears most strongly in specific distance and visibility regimes.

Example 2

Camera remains helpful in cases where LiDAR or radar support is weak, particularly when visual coverage is still good.

Example 3

Range sensors provide useful compensation when camera support is weak, showing that poor visual conditions do not always imply poor overall support.

Example 4

The strongest justification for fusion is not that all modalities are always useful together, but that different modalities remain informative in different failure regimes.

Example 5

Camera-dominant and range-dominant regimes both exist, which supports an adaptive fusion strategy instead of equal treatment of all modalities.

Strong concluding sentence template

Overall, the data shows clear cross-modal complementarity: camera can compensate for weak range sensing in some regimes, while LiDAR and radar can compensate for weak camera support in others, indicating that fusion value is conditional rather than uniform.

## Easy vs Hard Multimodal Regimes

### Key flags

In [119]:
# Strong across all sensors → easy cases
eda_df["strong_all_modalities"] = (
    eda_df["good_camera_view"] &
    eda_df["usable_lidar_support"] &
    eda_df["usable_radar_support"]
)

# Weak across all sensors → hard cases
eda_df["weak_all_modalities"] = (
    eda_df["weak_camera_view"] &
    ~eda_df["usable_lidar_support"] &
    ~eda_df["usable_radar_support"]
)

# Optional: intermediate cases (very useful)
eda_df["camera_only_strong"] = (
    eda_df["good_camera_view"] &
    ~eda_df["usable_lidar_support"] &
    ~eda_df["usable_radar_support"]
)

eda_df["range_only_strong"] = (
    eda_df["weak_camera_view"] &
    (
        eda_df["usable_lidar_support"] |
        eda_df["usable_radar_support"]
    )
)

### Global prevalence

In [ ]:
regime_flags = [
    "strong_all_modalities",
    "weak_all_modalities",
    "camera_only_strong",
    "range_only_strong",
]

regime_labels = {
    "strong_all_modalities": "Strong all modalities",
    "weak_all_modalities": "Weak all modalities",
    "camera_only_strong": "Camera only strong",
    "range_only_strong": "Range only strong",
}

regime_summary = (
    eda_df[regime_flags]
    .mean()
    .mul(100)
    .rename(index=regime_labels)
    .sort_values(ascending=False)
    .round(2)
)

print("Multimodal regime distribution (% of dataset):")
print(regime_summary)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
sns.barplot(
    x=regime_summary.values,
    y=regime_summary.index
)
plt.title("Prevalence of multimodal regimes")
plt.xlabel("Percentage of rows")
plt.ylabel("")
plt.tight_layout()
plt.show()

### By distance

In [ ]:
def summarize_flags_by_group(df, group_col, flag_cols):
    return (
        df.groupby(group_col)[flag_cols]
        .mean()
        .sort_index()
    )

regime_by_distance = summarize_flags_by_group(
    eda_df,
    "distance_bin",
    regime_flags
).rename(columns=regime_labels)

regime_by_distance.plot(kind="bar", figsize=(10, 5))
plt.title("Multimodal regimes by distance")
plt.xlabel("Distance bin")
plt.ylabel("Proportion")
plt.xticks(rotation=90)
plt.legend(
    title="",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

### By visibility

In [ ]:
regime_by_visibility = summarize_flags_by_group(
    eda_df,
    "visibility_level",
    regime_flags
).rename(columns=regime_labels)

regime_by_visibility.plot(kind="bar", figsize=(10, 5))
plt.title("Multimodal regimes by visibility")
plt.xlabel("Visibility level")
plt.ylabel("Proportion")
plt.xticks(rotation=90)
plt.legend(
    title="",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

### Category

In [ ]:
hard_by_category = (
    pd.crosstab(
        index=eda_df["category_name"],
        columns=eda_df["weak_all_modalities"],
        normalize="index",
    )
    .sort_values(by=True, ascending=False)
)

plot_bar_from_table(
    table=hard_by_category,
    column=True,
    title="Weak-all-modalities by category",
    ylabel="Proportion",
    xlabel="Category",
    rot=90
)

**Notes:**

This shows which classes are intrinsically hard even with fusion

### Observations

1. Easy cases
where all sensors are strong
likely close range / good visibility
2. Hard cases
where all sensors are weak
likely far + low visibility
3. Fusion usefulness
fusion helps mostly in intermediate regimes
fusion does NOT solve:
weak_all_modalities cases


Fusion is not universally effective; it is most beneficial in intermediate regimes where at least one modality remains informative, while fully weak regimes remain challenging.


What this section should conclude

This section should answer:

* where all sensors are strong
* where all sensors are weak
* how common easy and hard multimodal regimes are
* what that implies about the limits of fusion


Example key insights

Easy regimes
* A subset of cases is well supported across modalities, meaning they are likely easier and less dependent on sophisticated fusion.
* Strong-all-modality cases likely correspond to favorable distance and visibility conditions.

Hard regimes
* Weak-all-modality cases represent genuinely difficult situations where fusion may still struggle.
* These cases are especially important because they reveal the limits of sensor complementarity.
* If a meaningful share of the dataset falls into weak-all-modalities regimes, then improved fusion alone may not fully solve performance gaps.

Intermediate regimes
* Fusion is likely most valuable in intermediate support regimes, where at least one modality remains informative while another is weak.
* This suggests the best use of fusion is not in easiest cases or hardest cases, but in partially supported ones.


Strong concluding sentence template

Overall, multimodal support is polarized into easy, well-supported cases and hard, weakly supported cases, with fusion appearing most useful in intermediate regimes where at least one modality can compensate for another.

## Regime analysis: where does difficulty come from?

* How does detection difficulty change with distance?
* How does difficulty change with visibility / occlusion?
* How does difficulty change with object size or projected pixel area?
* Which combinations are especially hard: far + small, small + low visibility, far + low LiDAR support?
* Are there natural operating regimes in the data that should be analyzed separately?

# Machine learning

## Machine Learning Research Question
In this Jupyter notebook, I build a supervised 3D object detection pipeline on nuScenes by preparing inputs and targets directly from the official tables. For each sample (one timestamp in the sample table), I assemble the model input by collecting the linked sensor records in sample_data: the LiDAR point cloud file (LIDAR_TOP), the six camera image files (CAM_), and optionally the radar point cloud files (RADAR_). I use calibrated_sensor and ego_pose to transform all sensor measurements into a consistent coordinate frame (ego or global) and to align modalities spatially and temporally. I prepare the ground-truth output for each sample from sample_annotation by extracting category_name and the 3D bounding box parameters (translation, size, rotation) and the object velocity, forming a per-sample set of labeled 3D boxes. With these prepared inputs and outputs, I run a controlled ablation where the only change is which sample_data modalities are included (LiDAR-only, camera+LiDAR, camera+LiDAR+radar) and evaluate each variant using the official nuScenes metrics NDS, mAP, and AVE, followed by a short error breakdown by object class and distance.

## Planned Models and Rationale

## Detailed Machine Learning Strategy

# Key findings and conclusions